# Part 3: Decorators & Context Managers

Decorators and context managers are two powerful ways to add reusable behavior around existing code. Decorators wrap callables; context managers wrap a block of code and guarantee cleanup.

## Learning goals

By the end of this notebook, you should be able to:

- Build function, class, parameterized, nested, and stateful decorators.
- Preserve wrapped function metadata with `functools.wraps`.
- Explain decorator application and runtime execution order.
- Implement context managers with `with`, `__enter__`, and `__exit__`.
- Use `contextlib.contextmanager`, locks, files, database-like resources, transactions, and temporary resources safely.

> The key design question is: what behavior should happen before the operation, after it, and when it fails?

## 1. Decorators

A decorator is a callable that receives a function or class and returns a replacement or modified version. The `@decorator` syntax is executed when the decorated definition is created:

```python
@decorator
def work():
    pass

# Equivalent to:
# work = decorator(work)
```

The wrapper normally accepts `*args` and `**kwargs`, performs behavior around the original call, and returns its result. Use `functools.wraps` so the wrapper retains the wrapped function's `__name__`, docstring, annotations, and `__wrapped__` reference.

### Common decorator forms

- **Function decorator:** wraps a function to add logging, timing, authorization, caching, or retries.
- **Class decorator:** receives a class and can attach attributes or replace methods.
- **Parameterized decorator:** an outer function accepts configuration and returns the actual decorator.
- **Nested decorators:** several wrappers can be stacked. The closest decorator is applied first, but the outermost wrapper executes first at call time.
- **Stateful decorator:** stores state in a closure or callable object shared across calls to the decorated target.

For:

```python
@retry
@timed
@authenticated
def fetch():
    pass
```

Python builds the function as:

```python
fetch = retry(timed(authenticated(fetch)))
```

At call time, execution enters `retry`, then `timed`, then `authenticated`, then the original function. Return and exception handling unwind in the opposite direction. A retry decorator should usually wrap the operation that may fail, while authentication should run at the intended frequency, so ordering is a behavioral decision rather than just formatting.

In [6]:
from functools import wraps


def logged(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print(f"calling {function.__name__}")
        result = function(*args, **kwargs)
        print(f"finished {function.__name__}")
        return result

    return wrapper


@logged
def add(left, right):
    """Add two values."""
    return left + right


print("result:", add(2, 3))
print("preserved metadata:", add.__name__, add.__doc__)


def repeat(times):
    def decorate(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            results = []
            for _ in range(times):
                results.append(function(*args, **kwargs))
            return results

        return wrapper

    return decorate


@repeat(3)
def greet(name):
    return f"Hello, {name}!"


print("parameterized decorator:", greet("Ada"))


def announce(prefix):
    def decorate(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            print(prefix)
            return function(*args, **kwargs)

        return wrapper

    return decorate


@announce("outer")
@announce("inner")
def task():
    print("task body")
    return "done"


print("nested decorators:", task())


class CallCounter:
    def __init__(self, function):
        self.function = function
        self.call_count = 0
        wraps(function)(self)

    def __call__(self, *args, **kwargs):
        self.call_count += 1
        return self.function(*args, **kwargs)


def count_calls(function):
    return CallCounter(function)


@count_calls
def square(value):
    return value * value


square(4)
square(5)
print("stateful decorator count:", square.call_count)

calling add
finished add
result: 5
preserved metadata: add Add two values.
parameterized decorator: ['Hello, Ada!', 'Hello, Ada!', 'Hello, Ada!']
outer
inner
task body
nested decorators: done
stateful decorator count: 2


### Class decorators and a production-style decorator stack

A class decorator receives the class after its body has been executed. It can register the class, attach metadata, or return a replacement class. It is often simpler than a metaclass when the customization is local to one class definition.

Decorator stack order matters:

```python
@retry(attempts=3)
@timed
@authenticated

def fetch(user):
    ...
```

is equivalent to:

```python
fetch = retry(attempts=3)(timed(authenticated(fetch)))
```

At call time, `retry` is entered first. Each retry repeats everything inside it, including timing and authentication. If authentication should happen only once before all attempts, place it outside `retry`. Always make retry policies explicit: retry only transient exceptions, cap attempts, and use backoff and jitter in real network code.

In [7]:
from functools import wraps
from time import perf_counter


def add_tag(tag):
    def decorate(cls):
        cls.tag = tag
        return cls

    return decorate


@add_tag("billing")
class Invoice:
    pass


print("class decorator:", getattr(Invoice, "tag"))


def authenticated(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        user = kwargs.get("user") or (args[0] if args else None)
        if user != "admin":
            raise PermissionError("admin access required")
        print("authenticated")
        return function(*args, **kwargs)

    return wrapper


def timed(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        started = perf_counter()
        try:
            return function(*args, **kwargs)
        finally:
            elapsed = perf_counter() - started
            print(f"timed: {elapsed:.6f}s")

    return wrapper


def retry(attempts=3, exceptions=(RuntimeError,)):
    if attempts < 1:
        raise ValueError("attempts must be at least 1")

    def decorate(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            for attempt in range(1, attempts + 1):
                try:
                    return function(*args, **kwargs)
                except exceptions as error:
                    print(f"retry {attempt}/{attempts}: {error}")
                    if attempt == attempts:
                        raise

        return wrapper

    return decorate


attempts_seen = 0


@retry(attempts=3)
@timed
@authenticated
def fetch(user):
    global attempts_seen
    attempts_seen += 1
    if attempts_seen < 3:
        raise RuntimeError("temporary service failure")
    return f"data for {user}"


print("stacked decorators:", fetch("admin"))
print("attempts made:", attempts_seen)

try:
    fetch("guest")
except PermissionError as error:
    print("authentication failure:", error)

class decorator: billing
authenticated
timed: 0.000070s
retry 1/3: temporary service failure
authenticated
timed: 0.000022s
retry 2/3: temporary service failure
authenticated
timed: 0.000009s
stacked decorators: data for admin
attempts made: 3
timed: 0.000016s
authentication failure: admin access required


## 2. Context managers

A context manager controls setup and cleanup around a block:

```python
with resource() as value:
    use(value)
```

Conceptually, Python calls `__enter__`, runs the block, then calls `__exit__` even if the block raises an exception. The value returned by `__enter__` is bound after `as`; it is often the resource itself or a useful handle.

`__exit__(exc_type, exc_value, traceback)` must return a truthy value only when it intentionally suppresses the exception. Most managers return `False` or `None`, allowing errors to propagate. Cleanup belongs in `__exit__` or a `finally` block, not in code that runs only on success.

Use `contextlib.contextmanager` when setup and cleanup can be expressed clearly around one `yield`. `contextlib.closing`, `suppress`, `redirect_stdout`, `nullcontext`, and `ExitStack` provide reusable building blocks. For files, locks, database connections, and temporary resources, context managers make ownership and release boundaries visible.

A context manager is not limited to files: it can acquire a lock, begin a transaction, change a temporary environment, or establish and close a network/database-like connection.

In [8]:
from contextlib import contextmanager, suppress
from pathlib import Path
from tempfile import TemporaryDirectory
from threading import Lock


class DatabaseConnection:
    def __init__(self, name):
        self.name = name
        self.opened = False

    def __enter__(self):
        self.opened = True
        print(f"connected:{self.name}")
        return self

    def execute(self, query):
        if not self.opened:
            raise RuntimeError("connection is closed")
        return f"result for {query}"

    def __exit__(self, _exc_type, _exc_value, _traceback):
        self.opened = False
        print(f"closed:{self.name}")
        return False


with DatabaseConnection("analytics") as connection:
    print(connection.execute("SELECT 1"))
print("connection open after with:", connection.opened)


@contextmanager
def managed_resource(name):
    print(f"acquire:{name}")
    resource = {"name": name, "active": True}
    try:
        yield resource
    finally:
        resource["active"] = False
        print(f"release:{name}")


with managed_resource("cache") as resource:
    print("using resource:", resource["active"])
print("resource active after with:", resource["active"])


lock = Lock()
with lock:
    print("critical section owns lock:", lock.locked())
print("lock released:", lock.locked())


with TemporaryDirectory() as directory:
    file_path = Path(directory) / "notes.txt"
    file_path.write_text("context managers clean up resources", encoding="utf-8")
    with file_path.open(encoding="utf-8") as file:
        print("file content:", file.read())
    print("temporary file exists inside context:", file_path.exists())
print("temporary directory exists after context:", Path(directory).exists())


with suppress(FileNotFoundError):
    Path("does-not-exist.txt").unlink()
print("suppress handled the expected missing-file error")

connected:analytics
result for SELECT 1
closed:analytics
connection open after with: False
acquire:cache
using resource: True
release:cache
resource active after with: False
critical section owns lock: True
lock released: False
file content: context managers clean up resources
temporary file exists inside context: True
temporary directory exists after context: False
suppress handled the expected missing-file error


### Transactions and cleanup on failure

A transaction context manager can commit when the block completes normally and roll back when an exception escapes. The exception should usually be re-raised after rollback, so callers know the operation failed.

Nested context managers are entered from left to right and exited from right to left. This makes it possible to combine a transaction, a lock, and a temporary resource while preserving predictable cleanup order.

In [9]:
class Transaction:
    def __init__(self, storage):
        self.storage = storage
        self.snapshot = None

    def __enter__(self):
        self.snapshot = self.storage.copy()
        print("transaction started")
        return self.storage

    def __exit__(self, exc_type, exc_value, _traceback):
        if exc_type is None:
            print("transaction committed")
        else:
            self.storage.clear()
            self.storage.update(self.snapshot)
            print(f"transaction rolled back: {exc_value}")
        return False


account = {"balance": 100}
with Transaction(account) as state:
    state["balance"] += 50
print("after commit:", account)

try:
    with Transaction(account) as state:
        state["balance"] -= 25
        raise ValueError("payment rejected")
except ValueError:
    print("caller received transaction failure")
print("after rollback:", account)


from contextlib import ExitStack

with ExitStack() as stack:
    temporary = stack.enter_context(TemporaryDirectory())
    lock = Lock()
    stack.enter_context(lock)
    print("ExitStack resources active:", Path(temporary).exists(), lock.locked())
print("ExitStack completed cleanup")

transaction started
transaction committed
after commit: {'balance': 150}
transaction started
transaction rolled back: payment rejected
caller received transaction failure
after rollback: {'balance': 150}
ExitStack resources active: True True
ExitStack completed cleanup


## 3. Practice lab

Attempt each exercise before writing or running a solution.

### Decorator exercises

1. Write `debug` using `functools.wraps` that prints the function name, positional arguments, keyword arguments, and return value.
2. Write a class decorator `register` that adds each decorated class to a registry dictionary keyed by class name.
3. Write `repeat(times)` that rejects values below one and repeats the wrapped function exactly `times` times.
4. Build a stateful `count_calls` decorator whose wrapper exposes a read-only call count through a property or callable object.
5. Implement `retry` that retries only `ConnectionError`, then test it with a function that fails twice and succeeds on the third attempt.
6. Stack `@retry`, `@timed`, and `@authenticated`. Draw the equivalent nested function assignment and explain which behavior repeats on each retry.

### Context-manager exercises

7. Implement a class-based `Timer` context manager that reports elapsed time and still reports it if the block raises.
8. Rewrite `Timer` with `@contextmanager` and `try/finally`.
9. Create a `TemporaryDirectory` workflow that writes a file, reads it, and proves the directory is removed after the block.
10. Create a lock-protected counter and explain why `with lock:` is safer than manually calling `acquire` and `release`.
11. Build a transaction manager around a dictionary that commits on success and restores a snapshot on failure.
12. Use `ExitStack` to manage a variable number of files and ensure all files close if opening a later file fails.

### Review checklist

- [ ] I know `@decorator` is assignment syntax executed at definition time.
- [ ] I can distinguish decoration time from call time.
- [ ] I can explain why `wraps` matters for introspection and tooling.
- [ ] I can draw the execution order of stacked decorators.
- [ ] I retry only appropriate transient exceptions.
- [ ] I know `__exit__` runs on both success and failure.
- [ ] I return a truthy value from `__exit__` only when suppressing an exception intentionally.
- [ ] I use `finally` for cleanup that must happen regardless of success.
- [ ] I use context managers for resource ownership and explicit release boundaries.
- [ ] I can choose between a class-based manager, `@contextmanager`, and `ExitStack`.